# Offline toy pipeline: FASTQ → counts, with no primers supplied

> ⚠️ **Toy reproducibility example — _not_ benchmark evidence.**
> The demo library below is **synthetic**: its constants are planted, so
> `detect` recovering them only shows the pipeline *runs deterministically,
> end-to-end*. It says nothing about inference accuracy on real reads. For
> realistic `LibraryReport` behavior see **`02_library_report_interpretation.ipynb`**;
> for actual primer-recovery performance on paper-documented deposits see
> **Figure A** (`benchmarks/`).

This notebook runs the full `selexprep` local-FASTQ pipeline on a tiny,
self-generated demo library — **no download, no network, no HPC**. It is
deterministic (fixed RNG seed), so re-running gives identical results.

**Requirements:** `pip install selexprep` (which pulls in `cutadapt`).

**Pipeline:** `detect → extract → count → qc`. The thing `selexprep` does
that other tools don't: it *infers* the primer/constant regions from the
reads instead of requiring you to supply them.

## Calling the CLI from the notebook

`selexprep` is a command-line tool, so every stage below is a terminal
command. The `sh()` helper echoes the exact command it runs (the `$ …` line
you'd type yourself) and runs it with `check=True`, so a failed stage raises
and stops the notebook — i.e. executing this notebook is a genuine
end-to-end check, not a silent pass.

In [1]:
import shlex, subprocess


def sh(*args):
    """Run a selexprep CLI command, echoing the equivalent terminal line.

    Runs with ``check=True`` so a non-zero exit raises and aborts the notebook.
    """
    print("$ " + " ".join(shlex.quote(a) for a in args))
    proc = subprocess.run(
        args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=True
    )
    print(proc.stdout, end="")

## 1. Make a tiny demo library (no download)

A real run starts from an accession (`selexprep fetch <ACC>`, or just
`selexprep run accessions.tsv`). Here we fabricate two rounds of a
`5'const–N24–3'const` library so the notebook is self-contained and
reproducible.

In [2]:
import gzip, random, pathlib, shutil
random.seed(42)
P5, P3 = "GGAGCTCAGCCTTCAC", "CTGCAGTGACCTGAGT"   # constants detect must rediscover
rnd = lambda n: "".join(random.choice("ACGT") for _ in range(n))

demo = pathlib.Path("demo")
shutil.rmtree(demo, ignore_errors=True)   # start clean → re-running the notebook is idempotent
demo.mkdir()

def write(name, n, enrich=0.0):
    pool = [rnd(24) for _ in range(n)]
    with gzip.open(demo / name, "wt") as f:
        for i in range(n):
            N = pool[0] if (enrich and i < int(n * enrich)) else pool[i]
            seq = P5 + N + P3
            f.write(f"@read{i}\n{seq}\n+\n{'I' * len(seq)}\n")

write("demo_R1.fastq.gz", 800)               # round 1: diverse
write("demo_R3.fastq.gz", 800, enrich=0.4)   # round 3: one binder at 40% (selection signal)
(demo / "rounds.tsv").write_text(
    "file\tround_number\ndemo_R1.fastq.gz\t1\ndemo_R3.fastq.gz\t3\n"
)
print("wrote:", *(p.name for p in sorted(demo.iterdir())))

wrote: demo_R1.fastq.gz demo_R3.fastq.gz rounds.tsv


## 2. Infer the library structure — `detect`

`detect` reads the FASTQs (with a round map, so it can use cross-round
persistence) and emits a `LibraryReport`. Expect `status: HIGH`,
`extraction_mode: BOTH_PRIMERS_SINGLE_READ`, and recovered primers equal to
the constants we planted.

Because the constants are planted, this is a **plumbing check** — it shows
inference *runs* and is reproducible, not that it is accurate on messy real
reads (that evidence is Figure A).

In [3]:
sh("selexprep", "detect", "demo/demo_R1.fastq.gz", "demo/demo_R3.fastq.gz",
   "--round-map", "demo/rounds.tsv", "--outdir", "demo/out")

$ selexprep detect demo/demo_R1.fastq.gz demo/demo_R3.fastq.gz --round-map demo/rounds.tsv --outdir demo/out


library_report.json -> demo/out/library_report.json
  sha256:          1a669a696dd3...
  extraction_mode: BOTH_PRIMERS_SINGLE_READ
  required_action: NONE
  status:          HIGH


In [4]:
import json
rep = json.load(open("demo/out/library_report.json"))
for k in ["primer_5p", "primer_3p", "extraction_mode", "read_source",
          "required_action", "n_length_mode", "status", "confidence"]:
    print(f"{k:18} {rep[k]}")

primer_5p          GGAGCTCAGCCTTCAC
primer_3p          CTGCAGTGACCTGAGT
extraction_mode    BOTH_PRIMERS_SINGLE_READ
read_source        R1
required_action    NONE
n_length_mode      24
status             HIGH
confidence         1.0


`primer_5p` / `primer_3p` should equal `GGAGCTCAGCCTTCAC` /
`CTGCAGTGACCTGAGT` with `n_length_mode = 24` — recovered from the reads alone.

> **Safe failure:** below ~500 sequences in the earliest round, `detect`
> returns `status: UNABLE_TO_INFER` instead of guessing, and downstream
> `extract` refuses unless you pass `--override-primer-5p/-3p`.

## 3. Extract the random region — `extract`

`extract` trims the inferred constants (via `cutadapt`) and writes the N
region per round, plus a reproducibility manifest.

In [5]:
sh("selexprep", "extract", "demo/demo_R1.fastq.gz", "demo/demo_R3.fastq.gz",
   "--library-report", "demo/out/library_report.json",
   "--round-map", "demo/rounds.tsv", "--outdir", "demo/out")

$ selexprep extract demo/demo_R1.fastq.gz demo/demo_R3.fastq.gz --library-report demo/out/library_report.json --round-map demo/rounds.tsv --outdir demo/out


extract: wrote 4 files under demo/out
  demo/out/round_01/extracted.fasta.gz
  demo/out/round_03/extracted.fasta.gz
  demo/out/trim_reports.json
  demo/out/selexprep_manifest.json


## 4. Count per round — `count`

Counting both rounds surfaces the selection signal we planted.

In [6]:
sh("selexprep", "count", "demo/out/round_01/extracted.fasta.gz", "--round", "R1", "--outdir", "demo/out")
sh("selexprep", "count", "demo/out/round_03/extracted.fasta.gz", "--round", "R3", "--outdir", "demo/out")

$ selexprep count demo/out/round_01/extracted.fasta.gz --round R1 --outdir demo/out


counts.parquet -> demo/out/round_01/counts.parquet
  unique sequences: 800
  total reads:      800
  top sequence:     1 reads (1250 RPM)
  Shannon entropy:  9.64 bits
$ selexprep count demo/out/round_03/extracted.fasta.gz --round R3 --outdir demo/out


counts.parquet -> demo/out/round_03/counts.parquet
  unique sequences: 481
  total reads:      800
  top sequence:     320 reads (400000 RPM)
  Shannon entropy:  6.32 bits


Round 1 is near-uniform (Shannon entropy ~9.6 bits, top sequence ~0.1%);
round 3 collapses (entropy ~6.3 bits, top sequence ~40%) — the binder we
enriched.

## 5. QC — `qc`

`qc` reads the manifest + counts and writes four diagnostic plots plus
depth-aware suspicion flags.

In [7]:
sh("selexprep", "qc", "demo/out/selexprep_manifest.json")

$ selexprep qc demo/out/selexprep_manifest.json


qc: 1 flag(s) raised; 4 plot(s) written
  flags.yaml: demo/out/qc/flags.yaml
  plot: demo/out/qc/read_retention.png
  plot: demo/out/qc/primer_match_per_round.png
  plot: demo/out/qc/n_length_distribution.png
  plot: demo/out/qc/per_round_panel.png
  [WARN] low_total_reads


On this toy data you'll see a single `low_total_reads` flag (800 reads is
far below the 10k floor) — an honest QC signal, not an error. The four PNGs
land in `demo/out/qc/`.

## Next steps
- Run on a **real accession**: `selexprep run accessions.tsv --outdir out`
  (or `selexprep fetch <ACC>` then the same `detect → extract → count → qc`).
- See **`02_library_report_interpretation.ipynb`** to interpret `LibraryReport`
  fields on real benchmark deposits (high-confidence, partial, and
  safe-failure cases).